(dkist:examples:vbi-reproject)=

# Stitching a VBI Mosaic with `reproject`

```{note}
You will need the reproject and shapely packages installed to run this guide.

If you have installed `dkist` with pip you may need to run `pip install 'reproject[all]'` to install shapely as an optional dependency for reproject. If you installed with conda you may need to run `conda install shapely`.
```

The [reproject](https://reproject.readthedocs.io/) package is an Astropy-affiliated package for regridding data.
A number of different algorithms are implemented in the package, with different trade-offs for speed and accuracy.
Reprojecting a single spatial image such as an AIA image is well supported and demonstrated in the [sunpy gallery](https://docs.sunpy.org/en/latest/generated/gallery/index.html#combining-co-aligning-and-reprojecting-images).

We are going to use the example of using reproject's <a href="https://reproject.readthedocs.io/en/stable/api/reproject.mosaicking.reproject_and_coadd.html#reproject.mosaicking.reproject_and_coadd" target="_blank" style="text-decoration: underline">`reproject.mosaicking.reproject_and_coadd`</a> function to stitch a mosaic of VBI frames.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

import dkist
from dkist.data.sample import VBI_L1_NZJTB

## Obtaining some data

In this example we will use the VBI sample dataset [AJQWW](https://dkist.data.nso.edu/datasetview/AJQWW).
If you want to replace this dataset with your own dataset, see <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/howto_guides/sample_data.html#dkist-howto-guide-sample-data" target="_blank" style="text-decoration: underline">Downloading the Sample Data with Globus</a>.

Let's load the data with <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/api/dkist.load_dataset.html#dkist.load_dataset" target="_blank" style="text-decoration: underline">`dkist.load_dataset`</a>:


In [ ]:
ds = dkist.load_dataset(VBI_L1_NZJTB)
ds

This gives us a <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/api/dkist.TiledDataset.html#dkist.TiledDataset" target="_blank" style="text-decoration: underline">`dkist.TiledDataset`</a> object, which is an array of <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/api/dkist.Dataset.html#dkist.Dataset" target="_blank" style="text-decoration: underline">`dkist.Dataset`</a> objects, as this VBI dataset is tiled in space (or mosaiced).

The sample data includes the ASDF file along with the FITS files for the first frame in each mosaic position.

We can now make a composite plot of all the tiles, at the first timestep.


In [ ]:
fig = plt.figure(figsize=(12, 12))
fig = ds.plot(slice_index=0, share_zscale=True)

## Regridding with Reproject


In [ ]:
from reproject.mosaicking import find_optimal_celestial_wcs, reproject_and_coadd
from reproject import reproject_interp

from ndcube import NDCube

First, let us crop off the edges of all our tiles to remove some artifacts, and only select the first time step.
To do this we use the <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/api/dkist.TiledDataset.html#dkist.TiledDataset.slice_tiles" target="_blank" style="text-decoration: underline">`dkist.TiledDataset.slice_tiles`</a> helper which applies an array slice to each tile of the <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/api/dkist.TiledDataset.html#dkist.TiledDataset" target="_blank" style="text-decoration: underline">`dkist.TiledDataset`</a> object.


In [ ]:
first_tiles = ds.slice_tiles[0, 100:-100, 100:-100]

Next we need to calculate the optimal WCS for the output:


In [ ]:
reference_wcs, shape_out = find_optimal_celestial_wcs(
    [f.wcs for f in first_tiles.flat],
    auto_rotate=True,
    # We drop the output resolution by a factor of 10 to reduce memory
    # remove this line to run at the native resolution of the input data
    resolution=0.1*u.arcsec,
)

# Due to a bug in reproject we need to reverse the direction of the longitude axis
# https://github.com/astropy/reproject/issues/431
reference_wcs.wcs.cdelt[0] = -reference_wcs.wcs.cdelt[0]

Now we can do the actual reprojection


In [ ]:
arr, footprint = reproject_and_coadd(
    first_tiles.flat,
    reference_wcs,
    reproject_function=reproject_interp,
    shape_out=shape_out,
    roundtrip_coords=False,
)

Make a new `NDCube` object and plot it.


In [ ]:
plt.figure(figsize=(10,10))
stitched = NDCube(arr, reference_wcs)
stitched.plot()